In [ ]:
# %% [code]
# --- ONE-CLICK SETUP (CPE-only Edition) ---
# Downloads the setup environment from GitHub and executes it.
# Änderung: text2technique komplett entfernt.
# Nur noch Mistral-CPE + RAG Artifacts werden geladen.

import os
import sys
import subprocess

# =============================================
# 1. CONFIGURATION (Edit this part only!)
# =============================================
GITHUB_USER = "soctoiam"
REPO = "soc_to_iam"
BRANCH = "main"

SCRIPT_DIR = "tiir_process/runtime/runtimeFiles/"

BASE_URL = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO}/{BRANCH}/{SCRIPT_DIR}"

SETUP_SCRIPT = "setup_env.py"
RAW_URL = f"{BASE_URL}{SETUP_SCRIPT}"

# =============================================
# 2. EXECUTION LOGIC
# =============================================
if "SETUP_COMPLETED" in globals() and globals()["SETUP_COMPLETED"]:
    print("System is already set up and models are loaded.")
    print("   (To force a restart, restart the Kernel)")
else:
    print(f"Fetching setup script from: {REPO}...")
    try:
        subprocess.check_call(["wget", "-q", "-O", SETUP_SCRIPT, RAW_URL])

        print("Executing Setup (this takes ~5-8 mins)...")

        with open(SETUP_SCRIPT, "r") as f:
            exec(f.read(), globals())

        SETUP_COMPLETED = True
        print("\n DONE! Models are loaded in memory. You can now run inference.")

    except Exception as e:
        print(f"\n CRITICAL FAILURE: {e}")
        print("   Check your Internet connection or GitHub URL.")

In [ ]:
# %% [code]
# --- RUN FULL TIIR PROCESS (CPE-only Edition) ---
# Supports:
# 1. JSON File Input -> Parser -> CTI Object
# 2. Raw Text Input -> CPE Inference -> Orchestrator -> CTI Object
# 3. Loader Execution
#
# Pipeline: Input -> text2CPE -> Orchestrator -> Loader
# Process will take ~1min

import os
import json

# ==========================================
# 1. SCENARIO INPUT (Text OR File Path)
# ==========================================

# OPTION A.1: Path to json (fails, no CPE --> text2CPE runs)
# input_text = "attack_json_fail.json"

# OPTION A.2: Path to json (succeeds, CPE found --> skip inference)
# input_text = "attack_json_succ.json"

# OPTION B: Raw Text
input_text = """
  URGENT: Microsoft Windows Server 2022 Remote Code Execution (CVE-2024-30080).

  A critical vulnerability exists in the Microsoft Message Queuing (MSMQ) service
  on Microsoft Windows Server 2022. An attacker can send a specially crafted
  malicious MSMQ packet to a MSMQ server, resulting in remote code execution.
"""


SCRIPT_JSON_PARSER = "json_to_cti_parser.py"

# ==========================================
# 2. PRE-PROCESSING & ROUTING
# ==========================================
run_inference = True

if input_text.strip().endswith(".json") and os.path.exists(input_text.strip()):
    print(f"0. DETECTED JSON FILE: {input_text.strip()}")

    if os.path.exists(SCRIPT_JSON_PARSER):
        print("    -> Running JSON Parser...")

        json_file_path = input_text.strip()

        try:
            with open(SCRIPT_JSON_PARSER, "r") as f:
                exec(f.read(), globals())

            if os.path.exists("Test_STIX.json"):
                print("    JSON Parsed successfully. Skipping Inference.")
                run_inference = False
            else:
                print("    JSON Parsing produced text output. Proceeding to Inference.")

        except Exception as e:
            print(f"   Error running JSON Parser: {e}")
            print("    Falling back to raw text inference.")
    else:
        print(f"    Parser Script {SCRIPT_JSON_PARSER} not found.")

# ==========================================
# 3. TIIR EXECUTION LOGIC (CPE-only)
# ==========================================

if run_inference:
    first_line = input_text.strip().splitlines()[0] if input_text.strip() else "(empty)"
    print(f"1. STARTING CPE INFERENCE on:\n    '{first_line}...'")

    # A. Run CPE Extractor
    script_cpe = "text2CPE_inference.py"
    if os.path.exists(script_cpe):
        with open(script_cpe, "r") as f:
            exec(f.read(), globals())
    else:
        print(f"Script {script_cpe} missing. Run Setup first!")

    print("-" * 40)

    # B. Run Orchestrator (Builds Test_STIX.json)
    script_orch = "orchestrator_stix.py"
    if os.path.exists(script_orch):
        print("2. ORCHESTRATING (Building CTI Object)...")
        with open(script_orch, "r") as f:
            exec(f.read(), globals())
    else:
        print(f"Script {script_orch} missing!")

else:
    print("(Skipping Inference & Orchestration due to successful JSON Parsing)")

print("-" * 40)

# C. Run Loader (Updates CSVs)
script_loader = "Loader.py"
if os.path.exists(script_loader):
    print("3. LOADING (Updating Assets)...")

    if not os.path.exists("Accounts.CSV") or not os.path.exists("Permissions.CSV"):
        print("   WARNING: CSV files not found in working dir!")

    with open(script_loader, "r") as f:
        exec(f.read(), globals())
else:
    print(f"\u274c Script {script_loader} missing!")